# Untold Lores — WAN I2V Animation Server
Runs Wan2.1-I2V-14B (image→video) as an API server.

**Steps:**
1. Enable GPU: Settings → Accelerator → **GPU T4 x2**
2. Add your tokens to Kaggle Secrets: `HF_TOKEN`, `NGROK_TOKEN`
3. Run all cells in order
4. Copy `SVD_NGROK_URL=...` into your `.env` file
5. Run `npm run animate` on your laptop (after `npm run images`)

In [ ]:
# Cell 1 — Install dependencies
!pip install -q --upgrade diffusers huggingface_hub transformers accelerate flask pyngrok imageio[ffmpeg]
!pip show diffusers huggingface_hub | grep Version

In [ ]:
# Cell 2 — Login and load Wan2.1-I2V model
import torch
import io, os, tempfile
from PIL import Image
from diffusers import AutoencoderKLWan, WanImageToVideoPipeline
from diffusers.utils import export_to_video
from transformers import CLIPVisionModel

try:
    from kaggle_secrets import UserSecretsClient
    import huggingface_hub
    huggingface_hub.login(token=UserSecretsClient().get_secret('HF_TOKEN'), add_to_git_credential=False)
except Exception:
    pass

MODEL_ID = 'Wan-AI/Wan2.1-I2V-14B-480P-Diffusers'

print('Loading VAE...')
vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder='vae', torch_dtype=torch.float32)

print('Loading image encoder...')
image_encoder = CLIPVisionModel.from_pretrained(MODEL_ID, subfolder='image_encoder', torch_dtype=torch.float32)

print('Loading pipeline...')
pipe = WanImageToVideoPipeline.from_pretrained(
    MODEL_ID,
    vae=vae,
    image_encoder=image_encoder,
    torch_dtype=torch.bfloat16,
)
pipe.enable_model_cpu_offload()

TMP_DIR = tempfile.mkdtemp()
print(f'WAN I2V ready! TMP_DIR={TMP_DIR}')

In [ ]:
# Cell 3 — Start API server
from flask import Flask, request, send_file, jsonify
import threading, uuid

app = Flask(__name__)
jobs = {}

NEGATIVE_PROMPT = 'static, no motion, blurry, low quality, camera shake, text, watermark, ugly'

@app.route('/health')
def health():
    return jsonify({'status': 'ok', 'model': 'wan-i2v'})

@app.route('/img2vid/submit', methods=['POST'])
def submit():
    prompt = request.args.get('prompt') or 'subtle atmospheric motion, gentle breeze, soft fog drifting, no camera movement'
    seed = int(request.args.get('seed', 42))
    img_bytes = request.data
    if not img_bytes:
        return jsonify({'error': 'no image data in request body'}), 400

    job_id = str(uuid.uuid4())
    jobs[job_id] = {'status': 'processing'}

    def run():
        try:
            image = Image.open(io.BytesIO(img_bytes)).convert('RGB').resize((832, 480))
            generator = torch.Generator(device='cuda').manual_seed(seed)
            output = pipe(
                image=image,
                prompt=prompt,
                negative_prompt=NEGATIVE_PROMPT,
                height=480,
                width=832,
                num_frames=33,
                guidance_scale=5.0,
                generator=generator,
            )
            frames = output.frames[0]
            out_path = os.path.join(TMP_DIR, f'clip_{job_id}.mp4')
            export_to_video(frames, out_path, fps=16)
            jobs[job_id] = {'status': 'done', 'path': out_path}
        except Exception as e:
            import traceback
            traceback.print_exc()
            jobs[job_id] = {'status': 'error', 'error': str(e)}

    threading.Thread(target=run, daemon=True).start()
    return jsonify({'job_id': job_id})

@app.route('/img2vid/status/<job_id>')
def status(job_id):
    job = jobs.get(job_id)
    if not job:
        return jsonify({'status': 'not_found'}), 404
    return jsonify({'status': job['status'], 'error': job.get('error')})

@app.route('/img2vid/result/<job_id>')
def result(job_id):
    job = jobs.get(job_id)
    if not job or job['status'] != 'done':
        return jsonify({'error': 'not ready'}), 400
    out_path = job['path']
    del jobs[job_id]
    return send_file(out_path, mimetype='video/mp4')

threading.Thread(
    target=lambda: app.run(port=5000, use_reloader=False, threaded=True),
    daemon=True
).start()
print('Flask server started on port 5000')

In [ ]:
# Cell 4 — Expose with ngrok
from pyngrok import ngrok, conf

try:
    from kaggle_secrets import UserSecretsClient
    ngrok_token = UserSecretsClient().get_secret('NGROK_TOKEN')
except Exception:
    ngrok_token = 'PASTE_YOUR_NGROK_TOKEN_HERE'

conf.get_default().auth_token = ngrok_token
ngrok.kill()

public_url = ngrok.connect(5000).public_url
print('=' * 60)
print('WAN I2V server is live!')
print(f'SVD_NGROK_URL={public_url}')
print('Add that line to your .env, then run: npm run animate')
print('=' * 60)

In [ ]:
# Cell 5 — Keep session alive
import time
print('Server running... (interrupt kernel when done)')
while True:
    time.sleep(60)
    print('.', end='', flush=True)